# Solutions — ETL Basics: Extract → Transform → Aggregate → Load

Exercise notebook: `08_data_engineering_python/01_etl_basics.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Extract with lineage

**Working solution**

In [ ]:
import csv
from pathlib import Path
from typing import Iterator

def extract(path) -> Iterator[dict]:
    path = Path(path)
    with path.open(newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not any((v or "").strip() for v in row.values() if isinstance(v, str)):
                continue
            yield {**row, "_source": path.name, "_line": reader.line_num}

**Explanation**

Lineage columns make every downstream reject traceable to a file and line — the first thing you need when a business user asks 'where did this row come from?'.

**Complexity:** O(n), O(1) memory.

**Common mistakes**

- Reading the whole file into a list in the extract step.

## Exercise 2 — 🟡 Medium · A strict transform

**Working solution**

In [ ]:
from datetime import datetime, timezone

STATUSES = {"PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"}

def _required(raw, field):
    value = (raw.get(field) or "").strip()
    if not value:
        raise ValueError(f"{field} is required")
    return value

def _parse_ts(value, field):
    try:
        return datetime.fromisoformat(value.strip().replace("Z", "+00:00")).astimezone(timezone.utc)
    except (AttributeError, ValueError):
        raise ValueError(f"{field} is not an ISO timestamp: {value!r}") from None

def transform_order(raw: dict) -> dict:
    try:
        quantity = int(raw.get("quantity", ""))
    except ValueError:
        raise ValueError(f"quantity is not an integer: {raw.get('quantity')!r}") from None
    if quantity < 1:
        raise ValueError(f"quantity must be >= 1, got {quantity}")
    try:
        price = float(raw.get("unit_price", ""))
    except ValueError:
        raise ValueError(f"unit_price is not a number: {raw.get('unit_price')!r}") from None
    if price < 0:
        raise ValueError("unit_price must be >= 0")
    status = (raw.get("status") or "").strip().upper()
    if status not in STATUSES:
        raise ValueError(f"unknown status {status!r}")
    order_ts = _parse_ts(raw.get("order_ts"), "order_ts")
    record = {
        "order_id": _required(raw, "order_id"), "customer_id": _required(raw, "customer_id"),
        "product_id": _required(raw, "product_id"), "quantity": quantity, "unit_price": price,
        "amount": round(quantity * price, 2), "order_ts": order_ts, "order_date": order_ts.date().isoformat(),
        "status": status, "updated_at": _parse_ts(raw.get("updated_at"), "updated_at"),
        "channel": (raw.get("channel") or "").strip().lower(), "country": (raw.get("country") or "").strip().upper(),
    }
    for lineage in ("_source", "_line"):
        if lineage in raw:
            record[lineage] = raw[lineage]
    return record

**Explanation**

One function, one responsibility: raw → typed or exception. Small private helpers (`_required`, `_parse_ts`) keep it readable. Note that `status.lower()` values from the messy
file are *normalized*, not rejected — decide deliberately what's a fixable format issue versus a real data error.

**Complexity:** O(1) per row.

**Common mistakes**

- Returning partially-typed records with `None` everywhere instead of rejecting.
- Catching every exception and losing *why* the row failed.

## Exercise 3 — 🟡 Medium · Daily aggregates

**Working solution**

In [ ]:
from collections import defaultdict

def aggregate_daily(records):
    groups = defaultdict(lambda: {"order_ids": set(), "units": 0, "revenue": 0.0})
    for r in records:
        if r["status"] == "CANCELLED":
            continue
        g = groups[(r["order_date"], r["country"])]
        g["order_ids"].add(r["order_id"])
        g["units"] += r["quantity"]
        g["revenue"] += r["amount"]
    return [{"order_date": d, "country": c, "orders": len(g["order_ids"]), "units": g["units"], "revenue": round(g["revenue"], 2)}
            for (d, c), g in sorted(groups.items())]

**Explanation**

A `defaultdict` with a dict factory holds several accumulators per group. Counting **distinct** order ids (a set) avoids inflating counts when an order has multiple rows.

**Complexity:** O(n + g log g).

**Common mistakes**

- Counting rows as orders.
- Rounding inside the loop (accumulates rounding error).

## Exercise 4 — 🟠 Hard · Wire it together with rejects and stats

**Working solution**

In [ ]:
import csv
from pathlib import Path

CLEAN_FIELDS = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "amount", "order_ts", "order_date",
                "status", "updated_at", "channel", "country", "_source", "_line"]

def _iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def run_etl(input_path, output_dir):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = {"clean": out / "clean.csv", "rejected": out / "rejected.csv", "summary": out / "daily_summary.csv"}
    read = loaded = rejected = 0
    agg_inputs = []
    with paths["clean"].open("w", newline="", encoding="utf-8") as cf, paths["rejected"].open("w", newline="", encoding="utf-8") as rf:
        clean_w = csv.DictWriter(cf, fieldnames=CLEAN_FIELDS, extrasaction="ignore")
        clean_w.writeheader()
        rej_w = None
        for raw in extract(input_path):
            read += 1
            try:
                rec = transform_order(raw)
            except ValueError as exc:
                if rej_w is None:
                    rej_w = csv.DictWriter(rf, fieldnames=[*raw.keys(), "_error"], extrasaction="ignore")
                    rej_w.writeheader()
                rej_w.writerow({**raw, "_error": str(exc)})
                rejected += 1
                continue
            loaded += 1
            clean_w.writerow({**rec, "order_ts": _iso(rec["order_ts"]), "updated_at": _iso(rec["updated_at"])})
            agg_inputs.append({k: rec[k] for k in ("order_id", "order_date", "country", "quantity", "amount", "status")})
    summary = aggregate_daily(agg_inputs)
    with paths["summary"].open("w", newline="", encoding="utf-8") as sf:
        w = csv.DictWriter(sf, fieldnames=["order_date", "country", "orders", "units", "revenue"])
        w.writeheader()
        w.writerows(summary)
    return {"read": read, "loaded": loaded, "rejected": rejected,
            "reject_ratio": round(rejected / read, 4) if read else 0.0,
            "outputs": {k: str(v) for k, v in paths.items()}}

**Explanation**

The orchestrator is thin: it only wires steps, routes failures, and counts. Business logic stays in testable pure functions. Keeping a *slim projection* of each record
for aggregation (instead of whole rows) is the in-memory equivalent of Spark's column pruning.

**Complexity:** O(n) time; memory O(n × 6 small fields) for aggregation input.

**Common mistakes**

- Letting one bad row crash the run.
- No reject file → bad data silently disappears.

## Challenge — 🔴 Very Hard · Multi-file streaming ETL with a manifest

**Working solution**

In [ ]:
import csv
import hashlib
import json
import shutil
from pathlib import Path

PART_FIELDS = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "amount", "order_ts", "order_date",
               "status", "updated_at", "channel", "country", "_source", "_line"]

class _PartWriter:
    def __init__(self, directory, chunk_size):
        self.dir, self.chunk_size = Path(directory), chunk_size
        self.parts, self._f, self._w, self._rows = [], None, None, 0

    def write(self, row):
        if self._w is None or self._rows >= self.chunk_size:
            self._roll()
        self._w.writerow(row)
        self._rows += 1

    def _roll(self):
        self.close()
        path = self.dir / f"part-{len(self.parts):05d}.csv"
        self._f = path.open("w", newline="", encoding="utf-8")
        self._w = csv.DictWriter(self._f, fieldnames=PART_FIELDS, extrasaction="ignore")
        self._w.writeheader()
        self.parts.append({"path": path, "rows": 0})
        self._rows = 0

    def close(self):
        if self._f:
            self._f.close()
            self.parts[-1]["rows"] = self._rows
            self._f = None

def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def run_multi_file_etl(input_paths, output_dir, chunk_size=500):
    out = Path(output_dir)
    for sub in ("clean", "rejected"):
        shutil.rmtree(out / sub, ignore_errors=True)
        (out / sub).mkdir(parents=True)
    (out / "_SUCCESS").unlink(missing_ok=True)
    parts = _PartWriter(out / "clean", chunk_size)
    files = []
    try:
        for src in map(Path, input_paths):
            stats = {"source": src.name, "read": 0, "loaded": 0, "rejected": 0}
            rej_f = rej_w = None
            try:
                for raw in extract(src):
                    stats["read"] += 1
                    try:
                        rec = transform_order(raw)
                    except ValueError as exc:
                        if rej_w is None:
                            rej_f = (out / "rejected" / src.name).open("w", newline="", encoding="utf-8")
                            rej_w = csv.DictWriter(rej_f, fieldnames=[*raw.keys(), "_error"], extrasaction="ignore")
                            rej_w.writeheader()
                        rej_w.writerow({**raw, "_error": str(exc)})
                        stats["rejected"] += 1
                        continue
                    rec["order_ts"] = rec["order_ts"].strftime("%Y-%m-%dT%H:%M:%SZ")
                    rec["updated_at"] = rec["updated_at"].strftime("%Y-%m-%dT%H:%M:%SZ")
                    parts.write(rec)
                    stats["loaded"] += 1
            finally:
                if rej_f:
                    rej_f.close()
            files.append(stats)
    finally:
        parts.close()
    manifest = {
        "files": files,
        "parts": [{"path": p["path"].relative_to(out).as_posix(), "rows": p["rows"], "sha256": _sha256(p["path"])} for p in parts.parts],
        "totals": {k: sum(f[k] for f in files) for k in ("read", "loaded", "rejected")},
    }
    (out / "_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (out / "_SUCCESS").write_text("", encoding="utf-8")
    return manifest

**Explanation**

A tiny rolling writer class encapsulates "open a new part every N rows". The manifest makes the output self-describing (row counts + checksums for downstream validation),
and `_SUCCESS` written last is the same commit protocol Hadoop/Spark/Glue jobs use on S3.

**Complexity:** O(total rows), O(1) row memory.

**Common mistakes**

- Writing `_SUCCESS` first (consumers may read partial output).
- Appending to old part files on rerun.